# Boundary-Safe Bronze Text Preparation

This notebook reads one configured gzipped brWaC shard from `data/raw/brwac-clean-1.txt.gz`, splits literal `<END>` boundaries before extraction, preserves source text, derives normalized matching text, and writes bronze segments to Parquet.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "src" / "tal_qual").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/tal_qual")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("tal-qual-bronze-preparation")
    .getOrCreate()
)

spark.version

In [ ]:
from tal_qual import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, prepare_bronze_dataframe, write_bronze_parquet

input_path = REPO_ROOT / RAW_CORPUS_INPUT
output_path = REPO_ROOT / BRONZE_OUTPUT_PATH

input_path, output_path

In [ ]:
from pyspark.sql.functions import col, regexp_extract, substring

bronze_df = prepare_bronze_dataframe(spark, input_path)

bronze_df.printSchema()

preview_df = (
    bronze_df
    .select(
        regexp_extract(col("source_file"), r"([^/]+)$", 1).alias("source_file"),
        "original_line_id",
        "segment_id",
        substring("text_original", 1, 90).alias("text_original_preview"),
        substring("match_text", 1, 90).alias("match_text_preview"),
    )
    .limit(20)
)

display(preview_df.toPandas())

In [ ]:
bronze_df.count()

In [ ]:
write_bronze_parquet(bronze_df, output_path)

output_path